# Myntra Fashion Classifier

Upgrade from Fashion-MNIST (28×28 grayscale, 10 toy classes) to the **Myntra Fashion Product Images** dataset:
- 44,000 real RGB product photos
- 224×224 resolution
- Rich label hierarchy: masterCategory → subCategory → articleType
- Colour, season, gender metadata

We train and compare:
1. **MobileNetV2** — fast, mobile-friendly transfer learning
2. **EfficientNetV2-S** — higher accuracy, same training approach

And keep the full production pipeline: augmentation, multi-label heads, hierarchical fallback, evaluation, and export.

## 1. Setup

In [ ]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pathlib, json, os

SEED     = 42
IMG_SIZE = 96       # 96×96 — 3× better than Fashion-MNIST's 28×28, trains fast on CPU
BATCH    = 64
SUBSET   = 10000    # use 10K samples to keep runtime manageable on CPU
DATA_DIR = pathlib.Path('data')

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load & Explore the Data

In [ ]:
df = pd.read_csv(DATA_DIR / 'styles.csv', on_bad_lines='skip')

# Add image path and verify the file exists
df['image_path'] = df['id'].apply(lambda x: str(DATA_DIR / 'images' / f'{x}.jpg'))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Total valid samples : {len(df)}")
print(f"Columns             : {df.columns.tolist()}")
df.head(3)

In [ ]:
# Use subCategory as target — good balance between too broad (5 classes)
# and too granular (143 article types with severe class imbalance)
TARGET = 'subCategory'

# Keep classes with >= 200 samples for reliable training
counts   = df[TARGET].value_counts()
keep     = counts[counts >= 200].index
df       = df[df[TARGET].isin(keep)].reset_index(drop=True)

CLASS_NAMES = sorted(df[TARGET].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Samples after filter: {len(df)}")

In [ ]:
counts_filtered = df[TARGET].value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(counts_filtered.index, counts_filtered.values, color='steelblue', edgecolor='black')
ax.set_title(f'Class Distribution — {TARGET}')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Min: {counts_filtered.min()} | Max: {counts_filtered.max()}")

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for i, cls in enumerate(CLASS_NAMES[:18]):
    ax  = axes[i // 6][i % 6]
    row = df[df[TARGET] == cls].iloc[0]
    img = mpimg.imread(row['image_path'])
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Images per Class', y=1.01)
plt.tight_layout()
plt.show()

## 3. Prepare Dataset

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df[TARGET])

# Subsample to keep CPU training fast
df_sub = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), SUBSET // NUM_CLASSES), random_state=SEED)
).reset_index(drop=True)

print(f"Using {len(df_sub)} samples across {NUM_CLASSES} classes")

# Stratified split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(df_sub, test_size=0.3, stratify=df_sub['label'], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# Compute class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_df['label'].values
)
class_weight_dict = dict(enumerate(class_weights))

def make_dataset(dataframe, training=False):
    paths  = dataframe['image_path'].values
    labels = dataframe['label'].values

    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        if training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.15)
            img = tf.image.random_contrast(img, 0.85, 1.15)
            img = tf.image.random_saturation(img, 0.85, 1.15)
            img = tf.image.random_hue(img, 0.05)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

print(f"Train batches: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
# Visualise one training batch
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i >= len(images): ax.axis('off'); continue
    ax.imshow(images[i].numpy())
    ax.set_title(CLASS_NAMES[labels[i].numpy()], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Training Batch (with augmentation)', y=1.01)
plt.tight_layout()
plt.show()

## 4. Model 1 — MobileNetV2 Transfer Learning

Fast, lightweight, good for mobile/edge deployment.

In [ ]:
def build_mobilenet(num_classes):
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.mobilenet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='MobileNetV2')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model, base

mobilenet, mobilenet_base = build_mobilenet(NUM_CLASSES)
mobilenet.summary()

In [ ]:
history_mn = mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=1, monitor='val_loss'),
    ],
    verbose=2,
)

In [ ]:
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_mn_ft = mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
    ],
    verbose=2,
)

## 5. Model 2 — EfficientNetV2-S Transfer Learning

Higher accuracy than MobileNetV2 at similar speed. Better suited for server-side inference.

In [ ]:
def build_efficientnet(num_classes):
    base = applications.EfficientNetV2S(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.efficientnet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(512, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out, name='EfficientNetV2S')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model, base

efficientnet, efficientnet_base = build_efficientnet(NUM_CLASSES)
efficientnet.summary()

In [ ]:
history_en = efficientnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=1, monitor='val_loss'),
    ],
    verbose=2,
)

In [ ]:
efficientnet_base.trainable = True
for layer in efficientnet_base.layers[:-30]:
    layer.trainable = False

efficientnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_en_ft = efficientnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
    ],
    verbose=2,
)

## 6. Evaluate & Compare

In [ ]:
mn_loss, mn_acc = mobilenet.evaluate(test_ds, verbose=0)
en_loss, en_acc = efficientnet.evaluate(test_ds, verbose=0)

results = pd.DataFrame({
    'Model':      ['MobileNetV2', 'EfficientNetV2-S'],
    'Test Acc':   [mn_acc,  en_acc],
    'Test Loss':  [mn_loss, en_loss],
    'Params':     [mobilenet.count_params(), efficientnet.count_params()],
})
print(results.to_string(index=False))

In [ ]:
def merge_histories(*hists):
    merged = {}
    for h in hists:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

mn_hist = merge_histories(history_mn, history_mn_ft)
en_hist = merge_histories(history_en, history_en_ft)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax in zip(['accuracy', 'loss'], axes):
    ax.plot(mn_hist[metric],           label='MobileNetV2 — train')
    ax.plot(mn_hist[f'val_{metric}'],  label='MobileNetV2 — val')
    ax.plot(en_hist[metric],           label='EfficientNetV2-S — train', ls='--')
    ax.plot(en_hist[f'val_{metric}'],  label='EfficientNetV2-S — val',   ls='--')
    ax.set_title(metric.capitalize())
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_model = efficientnet if en_acc >= mn_acc else mobilenet
best_name  = 'EfficientNetV2-S' if en_acc >= mn_acc else 'MobileNetV2'

y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds  = best_model.predict(imgs, verbose=0).argmax(axis=1)
    y_pred.extend(preds)
    y_true.extend(lbls.numpy())

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
pd.DataFrame(report).T.round(3)

## 7. Inference on Real Product Images

In [ ]:
def predict_and_show(model, df_subset, n=10):
    sample = df_subset.sample(n, random_state=SEED)
    paths  = sample['image_path'].values
    true_labels = sample['label'].values

    imgs = np.stack([
        tf.image.resize(tf.cast(tf.image.decode_jpeg(
            tf.io.read_file(p), channels=3), tf.float32) / 255.0,
            [IMG_SIZE, IMG_SIZE]).numpy()
        for p in paths
    ])
    preds = model.predict(imgs, verbose=0)

    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    for i, ax in enumerate(axes.flat):
        pred_idx   = preds[i].argmax()
        confidence = preds[i][pred_idx]
        predicted  = CLASS_NAMES[pred_idx]
        actual     = CLASS_NAMES[true_labels[i]]
        color      = 'green' if pred_idx == true_labels[i] else 'red'
        ax.imshow(imgs[i])
        ax.set_title(f'Pred: {predicted} ({confidence:.0%})\nTrue: {actual}',
                     color=color, fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Real Product Predictions — {best_name}', y=1.01)
    plt.tight_layout()
    plt.show()

predict_and_show(best_model, test_df)

## 8. Multi-Label Head — Add Colour Prediction

Real e-commerce needs colour tags too. We add a second head to predict `baseColour` from the same backbone — no extra inference cost.

In [ ]:
colour_counts = df['baseColour'].value_counts()
top_colours   = colour_counts[colour_counts >= 200].index.tolist()
df_multi      = df_sub[df_sub['baseColour'].isin(top_colours)].copy().reset_index(drop=True)

le_colour = LabelEncoder()
df_multi['colour_label'] = le_colour.fit_transform(df_multi['baseColour'])
NUM_COLOURS  = len(le_colour.classes_)
COLOUR_NAMES = le_colour.classes_.tolist()

print(f"Colour classes ({NUM_COLOURS}): {COLOUR_NAMES}")
print(f"Multi-label samples: {len(df_multi)}")

In [ ]:
train_m, temp_m = train_test_split(df_multi, test_size=0.3, stratify=df_multi['label'], random_state=SEED)
val_m,  test_m  = train_test_split(temp_m,   test_size=0.5, stratify=temp_m['label'],  random_state=SEED)

def make_multi_dataset(dataframe, training=False):
    paths   = dataframe['image_path'].values
    cat_lbl = dataframe['label'].values
    col_lbl = dataframe['colour_label'].values

    def load(path, cat, col):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        if training:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.15)
            img = tf.image.random_contrast(img, 0.85, 1.15)
        return img, {'category': cat, 'colour': col}

    ds = tf.data.Dataset.from_tensor_slices((paths, cat_lbl, col_lbl))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_mds = make_multi_dataset(train_m, training=True)
val_mds   = make_multi_dataset(val_m)
test_mds  = make_multi_dataset(test_m)

In [ ]:
def build_multi_head(num_classes, num_colours):
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = applications.mobilenet_v2.preprocess_input(inp * 255.0)
    x   = base(x, training=False)
    emb = layers.GlobalAveragePooling2D(name='embedding')(x)

    cat = layers.BatchNormalization()(emb)
    cat = layers.Dropout(0.3)(cat)
    cat = layers.Dense(256, activation='relu')(cat)
    cat_out = layers.Dense(num_classes, activation='softmax', name='category')(cat)

    col = layers.Dense(128, activation='relu')(emb)
    col_out = layers.Dense(num_colours, activation='softmax', name='colour')(col)

    model = models.Model(inp, [cat_out, col_out], name='MultiHead_MobileNet')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={'category': 'sparse_categorical_crossentropy',
              'colour':   'sparse_categorical_crossentropy'},
        loss_weights={'category': 1.0, 'colour': 0.5},
        metrics={'category': 'accuracy', 'colour': 'accuracy'},
    )
    return model

multi_model = build_multi_head(NUM_CLASSES, NUM_COLOURS)
multi_model.summary()

history_multi = multi_model.fit(
    train_mds,
    validation_data=val_mds,
    epochs=4,
    callbacks=[
        callbacks.EarlyStopping(
            patience=2, restore_best_weights=True,
            monitor='val_category_accuracy', mode='max'),
    ],
    verbose=2,
)

results_multi = multi_model.evaluate(test_mds, verbose=0)
for name, val in zip(multi_model.metrics_names, results_multi):
    print(f'{name:40s}: {val:.4f}')

In [ ]:
sample = test_m.sample(6, random_state=SEED)
imgs = np.stack([
    tf.image.resize(tf.cast(tf.image.decode_jpeg(
        tf.io.read_file(p), channels=3), tf.float32) / 255.0,
        [IMG_SIZE, IMG_SIZE]).numpy()
    for p in sample['image_path']
])
cat_preds, col_preds = multi_model.predict(imgs, verbose=0)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    cat  = CLASS_NAMES[cat_preds[i].argmax()]
    col  = COLOUR_NAMES[col_preds[i].argmax()]
    true_cat = CLASS_NAMES[sample.iloc[i]['label']]
    true_col = sample.iloc[i]['baseColour']
    ax.set_title(f'Pred: {cat} | {col}\nTrue: {true_cat} | {true_col}', fontsize=8)
    ax.axis('off')
plt.suptitle('Multi-Head: Category + Colour Predictions', y=1.01)
plt.tight_layout()
plt.show()

## 9. Hierarchical Label Fallback

In [ ]:
# Build subCategory → masterCategory mapping from the dataset
COARSE_MAP = df.groupby('subCategory')['masterCategory'].agg(lambda x: x.mode()[0]).to_dict()
CONFIDENCE_THRESHOLD = 0.60

def predict_with_fallback(model, images, threshold=CONFIDENCE_THRESHOLD):
    probs = model.predict(images, verbose=0)
    # Handle multi-output models
    if isinstance(probs, list):
        probs = probs[0]
    results = []
    for p in probs:
        best_idx   = p.argmax()
        best_conf  = float(p[best_idx])
        fine_label = CLASS_NAMES[best_idx]
        if best_conf >= threshold:
            results.append({'label': fine_label, 'confidence': best_conf, 'fallback': False})
        else:
            coarse = COARSE_MAP.get(fine_label, 'Unknown')
            results.append({'label': coarse, 'confidence': best_conf, 'fallback': True})
    return results

# Demo
sample = test_df.sample(8, random_state=SEED)
imgs = np.stack([
    tf.image.resize(tf.cast(tf.image.decode_jpeg(
        tf.io.read_file(p), channels=3), tf.float32) / 255.0,
        [IMG_SIZE, IMG_SIZE]).numpy()
    for p in sample['image_path']
])
preds = predict_with_fallback(best_model, imgs)

fallback_df = pd.DataFrame([
    {'True': CLASS_NAMES[sample.iloc[i]['label']],
     'Prediction': r['label'],
     'Confidence': f"{r['confidence']:.1%}",
     'Used fallback': r['fallback']}
    for i, r in enumerate(preds)
])
print(fallback_df.to_string(index=False))

## 10. Save Models

In [ ]:
mobilenet.save('mobilenet_myntra.keras')
efficientnet.save('efficientnet_myntra.keras')
multi_model.save('multihead_myntra.keras')

# Save class metadata
meta = {
    'class_names':   CLASS_NAMES,
    'colour_names':  COLOUR_NAMES,
    'coarse_map':    COARSE_MAP,
    'num_classes':   NUM_CLASSES,
    'num_colours':   NUM_COLOURS,
    'img_size':      IMG_SIZE,
    'target':        TARGET,
}
with open('myntra_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved: mobilenet_myntra.keras')
print('Saved: efficientnet_myntra.keras')
print('Saved: multihead_myntra.keras')
print('Saved: myntra_meta.json')

## Summary

| Upgrade | Fashion-MNIST | Myntra Dataset |
|---|---|---|
| Images | 70,000 grayscale 28×28 | 44,000 RGB 224×224 |
| Classes | 10 (broad) | 20+ subCategories |
| Label richness | Category only | Category + Colour + Gender + Season |
| Real-world relevance | Low | High (actual product photos) |
| Models | Simple CNN + MobileNetV2 | MobileNetV2 + EfficientNetV2-S |
| Multi-label | Synthesised attributes | Real colour labels |

**Next steps:**
- Scale to DeepFashion (800K images) for higher accuracy
- Add gender and season as additional output heads
- Export best model to ONNX for cross-platform deployment
- Retrain the HF Spaces demo with the new model weights